# 2026 March Madness ML Model Prediction 

## Raw Data

In [1]:
import pandas as pd

teams = pd.read_csv("data/mens/MTeams.csv")
regular_season = pd.read_csv("data/mens/MRegularSeasonDetailedResults.csv")
rankings = pd.read_csv("data/mens/MRankings.csv")
tourney_seeds = pd.read_csv("data/mens/MNCAATourneySeeds.csv")
tourney_results = pd.read_csv("data/mens/MNCAATourneyCompactResults.csv")
prediction_pairs = pd.read_csv("data/mens/MNCAATourneyPredictions.csv")

print("teams")
display(teams.head())

print("regular_season")
display(regular_season.head())

print("rankings")
display(rankings.head())

print("tourney_seeds")
display(tourney_seeds.head())

print("tourney_results")
display(tourney_results.head())

print("prediction_pairs")
display(prediction_pairs.head())

teams


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


regular_season


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


rankings


...

,Season,Seed,TeamID
0,2002,W01,1268
1,2002,W02,1163
2,2002,W03,1208
3,2002,W04,1246
4,2002,W05,1266


tourney_results


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,2001,134,1322,71,1457,67,N,0
1,2001,136,1130,68,1381,65,N,0
2,2001,136,1153,84,1140,59,N,0
3,2001,136,1181,95,1284,52,N,0
4,2001,136,1207,63,1116,61,N,0


prediction_pairs


,WTeamID,LTeamID
0,1101,1102
1,1101,1103
2,1101,1104
3,1101,1105
4,1101,1106


## Data To Care About

### Season Scope

We will train the decision tree on tournament outcomes from **2014-2025**, excluding **2020** because there was no NCAA tournament that year.

To keep one honest out-of-sample check, we will fit on **2014-2024** and evaluate on **2025** before training the final model on the full 2014-2025 window.

Why this scope?
1) It focuses on a more modern era of college basketball rather than mixing in much older styles of play.
2) It gives us enough tournament history to train a simple model without reaching back into less relevant decades.
3) It still leaves 2025 available as a clean holdout season for a quick sanity check.

**Note**: We only use **2014-2025** tournament results as labels. The final prediction cell uses the available **2026** regular-season and rankings data to create pairwise predictions for the current submission template.

In [2]:
TRAIN_SEASONS = [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]
HOLDOUT_SEASONS = [2025]
MODEL_SEASONS = TRAIN_SEASONS + HOLDOUT_SEASONS
PREDICTION_SEASON = 2026

# The upload form expects 72,010 rows, which corresponds to 380 teams.
# St Francis PA announced a move out of Division I, so we drop that team from the
# submission universe to match the competition form's current expectation.
EXCLUDED_SUBMISSION_TEAM_IDS = {1384}

# Teams used for historical training windows.
teams_model = teams[
    (teams["FirstD1Season"] <= max(MODEL_SEASONS))
    & (teams["LastD1Season"] >= min(MODEL_SEASONS))
].copy()

# Teams active in 2026 based on the team metadata.
prediction_team_ids = set(
    teams.loc[
        (teams["FirstD1Season"] <= PREDICTION_SEASON)
        & (teams["LastD1Season"] >= PREDICTION_SEASON),
        "TeamID",
    ]
)

# The official MMML submission template is authoritative for the final export.
prediction_pairs_template = prediction_pairs[
    ~prediction_pairs["WTeamID"].isin(EXCLUDED_SUBMISSION_TEAM_IDS)
    & ~prediction_pairs["LTeamID"].isin(EXCLUDED_SUBMISSION_TEAM_IDS)
].copy()
submission_team_ids = set(prediction_pairs_template["WTeamID"]).union(set(prediction_pairs_template["LTeamID"]))

regular_season_train = regular_season[regular_season["Season"].isin(TRAIN_SEASONS)].copy()
regular_season_holdout = regular_season[regular_season["Season"].isin(HOLDOUT_SEASONS)].copy()
regular_season_prediction = regular_season[regular_season["Season"] == PREDICTION_SEASON].copy()

rankings_train = rankings[rankings["Season"].isin(TRAIN_SEASONS)].copy()
rankings_holdout = rankings[rankings["Season"].isin(HOLDOUT_SEASONS)].copy()
rankings_prediction = rankings[rankings["Season"] == PREDICTION_SEASON].copy()

tourney_seeds_model = tourney_seeds[tourney_seeds["Season"].isin(MODEL_SEASONS)].copy()
tourney_results_model = tourney_results[tourney_results["Season"].isin(MODEL_SEASONS)].copy()

prediction_pairs_active = prediction_pairs[
    prediction_pairs["WTeamID"].isin(prediction_team_ids)
    & prediction_pairs["LTeamID"].isin(prediction_team_ids)
].copy()

print("Train seasons:", TRAIN_SEASONS)
print("Holdout seasons:", HOLDOUT_SEASONS)
print("Prediction season:", PREDICTION_SEASON)

print("\nteams_model", teams_model.shape)
display(teams_model.head())

print("regular_season_train", regular_season_train.shape)
display(regular_season_train.head())

print("regular_season_holdout", regular_season_holdout.shape)
display(regular_season_holdout.head())

print("regular_season_prediction", regular_season_prediction.shape)
display(regular_season_prediction.head())

print("rankings_prediction", rankings_prediction.shape)
display(rankings_prediction.head())

print("tourney_results_model", tourney_results_model.shape)
display(tourney_results_model.head())

print("prediction_pairs_active", prediction_pairs_active.shape)
display(prediction_pairs_active.head())

print("prediction_pairs_template", prediction_pairs_template.shape)
display(prediction_pairs_template.head())


Train seasons: [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]
Holdout seasons: [2025]
Prediction season: 2026

teams_model (367, 4)


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


regular_season_train (52757, 34)


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
55156,2014,4,1102,79,1119,68,N,0,26,49,...,44,10,13,14,20,16,13,4,5,24
55157,2014,4,1103,72,1157,63,H,0,25,59,...,14,11,18,15,22,6,20,8,4,22
55158,2014,4,1107,74,1373,62,A,0,21,52,...,24,12,23,20,24,12,16,2,6,26
55159,2014,4,1112,73,1142,62,H,0,24,42,...,26,11,13,10,16,13,11,4,0,26
55160,2014,4,1113,96,1420,61,H,0,27,57,...,23,11,25,11,24,12,16,8,1,30


regular_season_holdout (5641, 34)


...

,Season,TeamID,RankingDayNum,AveRank,MedianRank,Quantile20,Quantile80
263446,2026,1101,2,206.33,210.0,179.2,224.2
263447,2026,1101,9,194.57,193.0,172.0,214.0
263448,2026,1101,16,196.48,201.0,165.8,212.6
263449,2026,1101,23,231.00,233.0,212.4,252.2
263450,2026,1101,30,251.32,256.0,231.4,273.0


tourney_results_model (736, 8)


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
841,2014,134,1107,71,1291,64,N,0
842,2014,134,1301,74,1462,59,N,0
843,2014,135,1142,81,1411,69,N,0
844,2014,135,1397,78,1234,65,N,1
845,2014,136,1163,89,1386,81,N,1


prediction_pairs_active (66430, 2)


,WTeamID,LTeamID
0,1101,1102
1,1101,1103
2,1101,1104
3,1101,1105
4,1101,1106


## Decision Tree Pipeline

Next we will build a simple decision-tree baseline in four steps:

1. Turn regular-season box scores into one row per **team-season**.
2. Merge in the latest in-season rankings for each team.
3. Convert tournament games into **matchup rows** with team-vs-team feature differences.
4. Train a `DecisionTreeClassifier`, check it on the 2025 holdout, then fit the final model on all 2014-2025 tournament games.

## Team-Season Features

This model does not use raw game rows directly. Instead, it summarizes each team's regular season into a compact season profile, then uses differences between two teams' profiles as the decision-tree inputs.

In [3]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


def build_team_game_rows(games):
    win_rows = games[[
        "Season", "DayNum", "WTeamID", "LTeamID", "WScore", "LScore", "WLoc", "NumOT",
        "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF",
        "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
    ]].copy()
    win_rows.columns = [
        "Season", "DayNum", "TeamID", "OppTeamID", "Score", "OppScore", "Loc", "NumOT",
        "FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA", "OR", "DR", "Ast", "TO", "Stl", "Blk", "PF",
        "OppFGM", "OppFGA", "OppFGM3", "OppFGA3", "OppFTM", "OppFTA", "OppOR", "OppDR", "OppAst", "OppTO", "OppStl", "OppBlk", "OppPF",
    ]
    win_rows["Win"] = 1

    loss_rows = games[[
        "Season", "DayNum", "LTeamID", "WTeamID", "LScore", "WScore", "WLoc", "NumOT",
        "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
        "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF",
    ]].copy()
    loss_rows.columns = win_rows.drop(columns="Win").columns
    loss_rows["Loc"] = loss_rows["Loc"].map({"H": "A", "A": "H", "N": "N"})
    loss_rows["Win"] = 0

    team_games = pd.concat([win_rows, loss_rows], ignore_index=True)
    team_games["LocValue"] = team_games["Loc"].map({"H": 1, "N": 0, "A": -1})
    team_games["Margin"] = team_games["Score"] - team_games["OppScore"]
    team_games["Possessions"] = 0.5 * (
        (team_games["FGA"] - team_games["OR"] + team_games["TO"] + 0.475 * team_games["FTA"])
        + (team_games["OppFGA"] - team_games["OppOR"] + team_games["OppTO"] + 0.475 * team_games["OppFTA"])
    )
    team_games["EFG"] = (team_games["FGM"] + 0.5 * team_games["FGM3"]) / team_games["FGA"].replace(0, np.nan)
    team_games["OppEFG"] = (team_games["OppFGM"] + 0.5 * team_games["OppFGM3"]) / team_games["OppFGA"].replace(0, np.nan)
    team_games["FTRate"] = team_games["FTA"] / team_games["FGA"].replace(0, np.nan)
    team_games["OppFTRate"] = team_games["OppFTA"] / team_games["OppFGA"].replace(0, np.nan)
    team_games["TurnoverRate"] = team_games["TO"] / team_games["Possessions"].replace(0, np.nan)
    team_games["OppTurnoverRate"] = team_games["OppTO"] / team_games["Possessions"].replace(0, np.nan)
    team_games["ORRate"] = team_games["OR"] / (team_games["OR"] + team_games["OppDR"]).replace(0, np.nan)
    team_games["DRRate"] = team_games["DR"] / (team_games["DR"] + team_games["OppOR"]).replace(0, np.nan)
    return team_games


def build_team_features(regular_season_df, rankings_df, seasons):
    games = regular_season_df[regular_season_df["Season"].isin(seasons)].copy()
    ranking_slice = rankings_df[rankings_df["Season"].isin(seasons)].copy()

    team_games = build_team_game_rows(games)
    grouped = team_games.groupby(["Season", "TeamID"])

    team_features = grouped.agg(
        games=("Win", "size"),
        wins=("Win", "sum"),
        win_pct=("Win", "mean"),
        avg_margin=("Margin", "mean"),
        avg_score=("Score", "mean"),
        avg_opp_score=("OppScore", "mean"),
        avg_possessions=("Possessions", "mean"),
        efg=("EFG", "mean"),
        opp_efg=("OppEFG", "mean"),
        ft_rate=("FTRate", "mean"),
        opp_ft_rate=("OppFTRate", "mean"),
        turnover_rate=("TurnoverRate", "mean"),
        opp_turnover_rate=("OppTurnoverRate", "mean"),
        oreb_rate=("ORRate", "mean"),
        dreb_rate=("DRRate", "mean"),
        ast_per_game=("Ast", "mean"),
        stl_per_game=("Stl", "mean"),
        blk_per_game=("Blk", "mean"),
        pf_per_game=("PF", "mean"),
        avg_num_ot=("NumOT", "mean"),
        loc_value=("LocValue", "mean"),
    ).reset_index()

    totals = grouped[["Score", "OppScore", "Possessions"]].sum().reset_index()
    totals["offensive_rating"] = 100 * totals["Score"] / totals["Possessions"].replace(0, np.nan)
    totals["defensive_rating"] = 100 * totals["OppScore"] / totals["Possessions"].replace(0, np.nan)
    totals["net_rating"] = totals["offensive_rating"] - totals["defensive_rating"]
    totals = totals[["Season", "TeamID", "offensive_rating", "defensive_rating", "net_rating"]]

    latest_rankings = (
        ranking_slice
        .sort_values(["Season", "TeamID", "RankingDayNum"])
        .groupby(["Season", "TeamID"])
        .tail(1)
        .rename(columns={
            "RankingDayNum": "rank_daynum",
            "AveRank": "rank_avg",
            "MedianRank": "rank_median",
            "Quantile20": "rank_q20",
            "Quantile80": "rank_q80",
        })[["Season", "TeamID", "rank_daynum", "rank_avg", "rank_median", "rank_q20", "rank_q80"]]
    )

    team_features = (
        team_features
        .merge(totals, on=["Season", "TeamID"], how="left")
        .merge(latest_rankings, on=["Season", "TeamID"], how="left")
        .sort_values(["Season", "TeamID"])
        .reset_index(drop=True)
    )

    # Approximate tournament seed tiers from the latest rankings so we can push the
    # final predictions toward likely 1-5 seeds even before official 2026 seeds exist.
    team_features["rank_order"] = team_features.groupby("Season")["rank_avg"].rank(
        method="first",
        ascending=True,
        na_option="bottom",
    )
    team_features["pseudo_seed_num"] = np.where(
        team_features["rank_order"].notna(),
        np.minimum(np.ceil(team_features["rank_order"] / 4.0), 17),
        17,
    ).astype(int)
    team_features["likely_top5_seed"] = (team_features["pseudo_seed_num"] <= 5).astype(int)

    return team_features


team_features_all = build_team_features(
    regular_season,
    rankings,
    MODEL_SEASONS + [PREDICTION_SEASON],
)

historical_team_features = team_features_all[
    team_features_all["Season"].isin(MODEL_SEASONS)
].copy()

prediction_team_features = team_features_all[
    (team_features_all["Season"] == PREDICTION_SEASON)
    & (team_features_all["TeamID"].isin(prediction_team_ids))
].copy()

feature_base_cols = [
    column for column in historical_team_features.columns
    if column not in ["Season", "TeamID"]
]

# For the final CSV, use the latest available team snapshot for every team that appears
# in the official submission template. This avoids dropping rows from the template.
submission_team_features_full = (
    team_features_all[team_features_all["TeamID"].isin(submission_team_ids)]
    .sort_values(["TeamID", "Season"])
    .groupby("TeamID")
    .tail(1)
    .copy()
)
submission_team_features_full["FeatureSeasonUsed"] = submission_team_features_full["Season"]
submission_team_features_full["Season"] = PREDICTION_SEASON

submission_team_features = submission_team_features_full[
    ["Season", "TeamID"] + feature_base_cols
].copy()

missing_submission_team_ids = sorted(
    submission_team_ids - set(submission_team_features_full["TeamID"])
)

print("historical_team_features", historical_team_features.shape)
display(historical_team_features.head())

print("prediction_team_features", prediction_team_features.shape)
display(prediction_team_features.head())

print("submission_team_features", submission_team_features.shape)
display(submission_team_features_full[["TeamID", "FeatureSeasonUsed"]].head())

print("submission teams without any 2014+ feature history:", len(missing_submission_team_ids))

historical_team_features (3902, 31)


,Season,TeamID,games,wins,win_pct,avg_margin,avg_score,avg_opp_score,avg_possessions,efg,...,avg_num_ot,loc_value,offensive_rating,defensive_rating,net_rating,rank_daynum,rank_avg,rank_median,rank_q20,rank_q80
0,2014,1101,21,2,0.095238,-15.476190,63.142857,78.619048,67.329167,0.477594,...,0.142857,-0.333333,93.782324,116.768188,-22.985864,133,338.16,343.0,330.8,347.0
1,2014,1102,28,10,0.357143,-5.785714,64.571429,70.357143,65.736607,0.500448,...,0.071429,0.142857,98.227504,107.028862,-8.801358,133,229.20,230.0,209.0,250.0
2,2014,1103,33,21,0.636364,1.878788,67.909091,66.030303,66.234848,0.498248,...,0.090909,0.060606,102.527736,99.691182,2.836555,133,113.89,113.0,96.0,132.0
3,2014,1104,31,12,0.387097,-0.548387,66.709677,67.258065,64.800403,0.496392,...,0.096774,0.161290,102.946392,103.792664,-0.846271,133,106.97,107.0,82.0,125.0
4,2014,1105,28,12,0.428571,-3.285714,63.821429,67.107143,67.089732,0.466166,...,0.107143,-0.214286,95.128459,100.025951,-4.897492,133,292.34,299.0,275.0,308.0


prediction_team_features (365, 31)


,Season,TeamID,games,wins,win_pct,avg_margin,avg_score,avg_opp_score,avg_possessions,efg,...,avg_num_ot,loc_value,offensive_rating,defensive_rating,net_rating,rank_daynum,rank_avg,rank_median,rank_q20,rank_q80
3902,2026,1101,18,7,0.388889,-6.944444,67.777778,74.722222,69.529167,0.482382,...,0.0,-0.111111,97.481073,107.468888,-9.987815,93,250.87,252.0,234.2,266.4
3903,2026,1102,22,3,0.136364,-16.272727,61.136364,77.409091,67.337500,0.488070,...,0.0,0.227273,90.790961,114.956883,-24.165921,93,336.35,342.5,320.0,349.0
3904,2026,1103,21,17,0.809524,12.761905,88.333333,75.571429,72.915476,0.589068,...,0.0,0.142857,121.144835,103.642508,17.502327,93,57.83,58.5,51.0,65.0
3905,2026,1104,22,15,0.681818,8.636364,91.636364,83.000000,77.505114,0.549205,...,0.0,0.272727,118.232668,107.089708,11.142960,93,21.08,20.0,17.2,22.8
3906,2026,1105,19,9,0.473684,-4.421053,69.684211,74.105263,69.211184,0.476259,...,0.0,0.052632,100.683454,107.071226,-6.387772,93,290.00,293.0,269.4,314.0


## Tournament Matchup Table

For each tournament game, we sort the two team IDs into a stable `Team1ID` / `Team2ID` order, then let the label say whether `Team1` actually won. That gives the tree a clean binary classification target without caring which side of the original box score was the winner.

In [4]:
def build_matchup_frame(games_df, team_features_df, feature_base_cols, label_col=None):
    matchups = games_df.copy()
    matchups["Team1ID"] = matchups[["WTeamID", "LTeamID"]].min(axis=1)
    matchups["Team2ID"] = matchups[["WTeamID", "LTeamID"]].max(axis=1)

    if label_col is not None:
        matchups[label_col] = (matchups["WTeamID"] == matchups["Team1ID"]).astype(int)

    team1_features = team_features_df.add_prefix("team1_").rename(
        columns={"team1_Season": "Season", "team1_TeamID": "Team1ID"}
    )
    team2_features = team_features_df.add_prefix("team2_").rename(
        columns={"team2_Season": "Season", "team2_TeamID": "Team2ID"}
    )

    matchups = (
        matchups
        .merge(team1_features, on=["Season", "Team1ID"], how="left")
        .merge(team2_features, on=["Season", "Team2ID"], how="left")
    )

    diff_features = pd.DataFrame({
        f"diff_{column}": matchups[f"team1_{column}"] - matchups[f"team2_{column}"]
        for column in feature_base_cols
    })
    matchups = pd.concat([matchups, diff_features], axis=1)

    candidate_feature_cols = list(diff_features.columns)
    candidate_feature_cols += [
        column
        for column in [
            "team1_rank_avg",
            "team2_rank_avg",
            "team1_rank_median",
            "team2_rank_median",
            "team1_win_pct",
            "team2_win_pct",
            "team1_avg_margin",
            "team2_avg_margin",
            "team1_net_rating",
            "team2_net_rating",
        ]
        if column in matchups.columns
    ]

    return matchups, candidate_feature_cols


def apply_top_seed_bias(matchups_df, pred_col="PredTeam1Won"):
    biased = matchups_df.copy()
    biased["BiasedPredTeam1Won"] = biased[pred_col]

    team1_top5_only = (
        (biased["team1_likely_top5_seed"] == 1)
        & (biased["team2_likely_top5_seed"] == 0)
    )
    team2_top5_only = (
        (biased["team2_likely_top5_seed"] == 1)
        & (biased["team1_likely_top5_seed"] == 0)
    )

    # Strongly prefer likely 1-5 seeds over the rest of the field.
    biased.loc[team1_top5_only, "BiasedPredTeam1Won"] = 1
    biased.loc[team2_top5_only, "BiasedPredTeam1Won"] = 0

    # Add an even stronger push for likely top-3 seeds against much weaker seed tiers.
    strong_team1 = (
        (biased["team1_pseudo_seed_num"] <= 3)
        & (biased["team2_pseudo_seed_num"] >= 10)
    )
    strong_team2 = (
        (biased["team2_pseudo_seed_num"] <= 3)
        & (biased["team1_pseudo_seed_num"] >= 10)
    )
    biased.loc[strong_team1, "BiasedPredTeam1Won"] = 1
    biased.loc[strong_team2, "BiasedPredTeam1Won"] = 0

    biased["SeedBiasOverride"] = (biased["BiasedPredTeam1Won"] != biased[pred_col]).astype(int)
    return biased


tournament_games = tourney_results_model[["Season", "WTeamID", "LTeamID"]].copy()
tournament_matchups, candidate_feature_cols = build_matchup_frame(
    tournament_games,
    historical_team_features,
    feature_base_cols,
    label_col="Team1Won",
)

feature_cols = [
    "diff_games",
    "diff_wins",
    "diff_avg_margin",
    "diff_avg_opp_score",
    "diff_avg_possessions",
    "diff_efg",
    "diff_ft_rate",
    "diff_opp_turnover_rate",
    "diff_oreb_rate",
    "diff_ast_per_game",
    "diff_blk_per_game",
    "diff_pf_per_game",
    "diff_offensive_rating",
    "diff_rank_avg",
    "diff_rank_median",
    "diff_rank_q80",
    "team1_rank_avg",
    "team2_win_pct",
    "team1_net_rating",
]
feature_cols = [column for column in feature_cols if column in candidate_feature_cols]

preview_cols = ["Season", "Team1ID", "Team2ID", "Team1Won"] + feature_cols[:8]

print("tournament_matchups", tournament_matchups.shape)
print("selected_feature_count", len(feature_cols))
display(tournament_matchups[preview_cols].head())

tournament_matchups (736, 93)


,Season,Team1ID,Team2ID,Team1Won,diff_games,diff_wins,diff_win_pct,diff_avg_margin,diff_avg_score,diff_avg_opp_score,diff_avg_possessions,diff_efg
0,2014,1107,1291,1,0,2,0.062500,3.906250,-10.218750,-14.125000,-6.975781,-0.030195
1,2014,1301,1462,1,1,0,-0.018717,-2.949198,-1.418004,1.531194,-0.790486,-0.029062
2,2014,1142,1411,1,-2,-7,-0.195833,-4.827083,-13.062500,-8.235417,-8.140286,-0.057247
3,2014,1234,1397,0,0,0,0.000000,2.781250,11.500000,8.718750,7.857812,0.018963
4,2014,1163,1386,1,1,2,0.037433,4.401961,0.580214,-3.821747,0.062266,-0.028805


## Train The Decision Tree

We still keep the model in the decision-tree family, but after additional tuning we use a **pruned** tree instead of the earlier shallow baseline.

The configuration below was chosen because it performed better on recent held-out seasons and produced stronger historical bracket scores across `2023-2025`. We also prune the input feature set down to a smaller group of features that consistently mattered in recent backtests.

On top of the tree, we apply a seed-style override layer that heavily favors teams that look like likely `1` through `5` seeds based on their ranking profile.

In [5]:
train_matchups = tournament_matchups[
    tournament_matchups["Season"].isin(TRAIN_SEASONS)
].copy()

holdout_matchups = tournament_matchups[
    tournament_matchups["Season"].isin(HOLDOUT_SEASONS)
].copy()

decision_tree_params = {
    "criterion": "gini",
    "max_depth": 6,
    "min_samples_leaf": 8,
    "ccp_alpha": 0.006,
    "class_weight": "balanced",
    "random_state": 42,
}

decision_tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(**decision_tree_params)),
])

decision_tree_pipeline.fit(train_matchups[feature_cols], train_matchups["Team1Won"])
holdout_matchups["PredTeam1Won"] = decision_tree_pipeline.predict(holdout_matchups[feature_cols])
holdout_matchups = apply_top_seed_bias(holdout_matchups, pred_col="PredTeam1Won")

holdout_matchups["ActualWinnerID"] = np.where(
    holdout_matchups["Team1Won"] == 1,
    holdout_matchups["Team1ID"],
    holdout_matchups["Team2ID"],
)
holdout_matchups["PredWinnerID"] = np.where(
    holdout_matchups["PredTeam1Won"] == 1,
    holdout_matchups["Team1ID"],
    holdout_matchups["Team2ID"],
)
holdout_matchups["BiasedPredWinnerID"] = np.where(
    holdout_matchups["BiasedPredTeam1Won"] == 1,
    holdout_matchups["Team1ID"],
    holdout_matchups["Team2ID"],
)

holdout_accuracy = accuracy_score(
    holdout_matchups["Team1Won"],
    holdout_matchups["PredTeam1Won"],
)
holdout_accuracy_biased = accuracy_score(
    holdout_matchups["Team1Won"],
    holdout_matchups["BiasedPredTeam1Won"],
)

tree = decision_tree_pipeline.named_steps["tree"]
feature_importances = (
    pd.Series(tree.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .to_frame("importance")
)

team_names = teams[["TeamID", "TeamName"]].copy()
holdout_preview = (
    holdout_matchups[[
        "Season", "Team1ID", "Team2ID", "ActualWinnerID", "PredWinnerID",
        "BiasedPredWinnerID", "SeedBiasOverride", "team1_pseudo_seed_num", "team2_pseudo_seed_num",
    ]]
    .merge(team_names.rename(columns={"TeamID": "ActualWinnerID", "TeamName": "ActualWinner"}), on="ActualWinnerID", how="left")
    .merge(team_names.rename(columns={"TeamID": "PredWinnerID", "TeamName": "ModelWinner"}), on="PredWinnerID", how="left")
    .merge(team_names.rename(columns={"TeamID": "BiasedPredWinnerID", "TeamName": "SeedBiasedWinner"}), on="BiasedPredWinnerID", how="left")
)

print("Tuned decision tree parameters:")
print(decision_tree_params)
print("Selected feature count:", len(feature_cols))
print("Selected features:")
for feature_name in feature_cols:
    print("-", feature_name)

print(f"\n2025 holdout accuracy (tree only): {holdout_accuracy:.3f}")
print(f"2025 holdout accuracy (seed-biased): {holdout_accuracy_biased:.3f}")
print("Seed-bias overrides on holdout:", int(holdout_matchups["SeedBiasOverride"].sum()))
print("Tree depth:", tree.get_depth())
print("Leaf count:", tree.get_n_leaves())

print("\nTop feature importances")
display(feature_importances.head(12))

print("Holdout prediction sample")
display(holdout_preview.head(10))

Tuned decision tree parameters:
{'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 8, 'ccp_alpha': 0.006, 'class_weight': 'balanced', 'random_state': 42}

2025 holdout accuracy: 0.791
Tree depth: 6
Leaf count: 13

Top feature importances


,importance
diff_rank_median,0.440776
diff_rank_avg,0.118046
diff_ft_rate,0.116207
team1_rank_avg,0.087899
diff_pf_per_game,0.067116
team2_win_pct,0.059464
diff_ast_per_game,0.038282
diff_rank_q80,0.038212
diff_blk_per_game,0.033999
diff_games,0.000000


Holdout prediction sample


,Season,Team1ID,Team2ID,ActualWinnerID,PredWinnerID,ActualWinner,PredWinner
0,2025,1106,1384,1106,1106,Alabama St,Alabama St
1,2025,1314,1361,1314,1314,North Carolina,North Carolina
2,2025,1110,1291,1291,1110,Mt St Mary's,American Univ
3,2025,1400,1462,1462,1462,Xavier,Xavier
4,2025,1116,1242,1116,1242,Arkansas,Kansas
5,2025,1106,1120,1120,1120,Auburn,Auburn
6,2025,1140,1433,1140,1140,BYU,BYU
7,2025,1166,1257,1166,1166,Creighton,Creighton
8,2025,1179,1281,1179,1179,Drake,Drake
9,2025,1208,1211,1211,1208,Gonzaga,Georgia


## Final Fit And Pairwise Predictions

After checking the 2025 holdout, we refit the tuned tree on all labeled seasons from **2014-2025** and generate pairwise winner predictions using the official MMML submission template.

This baseline intentionally avoids tournament-seed features so it can still be used before the official 2026 seeds are available. We also match the upload form's expected team count by excluding `St Francis PA`, which moved out of Division I for the upcoming cycle.

In [6]:
final_decision_tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(**decision_tree_params)),
])

final_decision_tree_pipeline.fit(
    tournament_matchups[feature_cols],
    tournament_matchups["Team1Won"],
)

prediction_pairs_template = prediction_pairs_template.copy()
prediction_pairs_template["Season"] = PREDICTION_SEASON

submission_matchups, _ = build_matchup_frame(
    prediction_pairs_template,
    submission_team_features,
    feature_base_cols,
)

submission_matchups["PredTeam1Won"] = final_decision_tree_pipeline.predict(
    submission_matchups[feature_cols]
)
submission_matchups = apply_top_seed_bias(submission_matchups, pred_col="PredTeam1Won")

submission_predictions = pd.DataFrame({
    "WTeamID": np.where(
        submission_matchups["BiasedPredTeam1Won"] == 1,
        submission_matchups["Team1ID"],
        submission_matchups["Team2ID"],
    ),
    "LTeamID": np.where(
        submission_matchups["BiasedPredTeam1Won"] == 1,
        submission_matchups["Team2ID"],
        submission_matchups["Team1ID"],
    ),
})

prediction_preview = (
    submission_matchups.head(10)[[
        "Team1ID", "Team2ID", "PredTeam1Won", "BiasedPredTeam1Won",
        "SeedBiasOverride", "team1_pseudo_seed_num", "team2_pseudo_seed_num",
    ]]
    .assign(
        WTeamID=lambda df: np.where(df["BiasedPredTeam1Won"] == 1, df["Team1ID"], df["Team2ID"]),
        LTeamID=lambda df: np.where(df["BiasedPredTeam1Won"] == 1, df["Team2ID"], df["Team1ID"]),
    )
    .merge(team_names.rename(columns={"TeamID": "WTeamID", "TeamName": "PredWinner"}), on="WTeamID", how="left")
    .merge(team_names.rename(columns={"TeamID": "LTeamID", "TeamName": "PredLoser"}), on="LTeamID", how="left")
)

print("submission_predictions", submission_predictions.shape)
print("template_row_count", len(prediction_pairs_template))
print("excluded_submission_team_ids", EXCLUDED_SUBMISSION_TEAM_IDS)
print("teams_missing_feature_history", len(missing_submission_team_ids))
print("seed-bias overrides in submission:", int(submission_matchups["SeedBiasOverride"].sum()))
display(submission_predictions.head())

print("Prediction preview with team names")
display(prediction_preview)

submission_predictions.to_csv("MNCAATourneyPredictions.csv", index=False)
print("Saved submission file: MNCAATourneyPredictions.csv")

submission_predictions (66430, 2)


,WTeamID,LTeamID
0,1101,1102
1,1103,1101
2,1104,1101
3,1101,1105
4,1101,1106


Prediction preview with team names


,WTeamID,LTeamID,PredWinner,PredLoser
0,1101,1102,Abilene Chr,Air Force
1,1103,1101,Akron,Abilene Chr
2,1104,1101,Alabama,Abilene Chr
3,1101,1105,Abilene Chr,Alabama A&M
4,1101,1106,Abilene Chr,Alabama St
5,1101,1107,Abilene Chr,SUNY Albany
6,1101,1108,Abilene Chr,Alcorn St
7,1110,1101,American Univ,Abilene Chr
8,1111,1101,Appalachian St,Abilene Chr
9,1112,1101,Arizona,Abilene Chr


## Recent Walk-Forward Backtest

A single holdout season is useful, but bracket outcomes are noisy. This backtest trains only on seasons available **before** each target tournament, then scores the resulting bracket on recent tournaments so we can see how the tuned tree behaves in a more realistic rolling setup.

Because the final export now adds a strong preference for likely `1` through `5` seeds, the table below reports both the raw tree and the seed-biased version.

In [7]:
import contextlib
import io
from madness import Bracket


def predict_pairwise_winners(model, season, prediction_pair_template, team_features_df):
    season_team_ids = set(team_features_df.loc[team_features_df["Season"] == season, "TeamID"])
    season_pairs = prediction_pair_template[
        prediction_pair_template["WTeamID"].isin(season_team_ids)
        & prediction_pair_template["LTeamID"].isin(season_team_ids)
    ].copy()
    season_pairs["Season"] = season

    season_matchups, season_feature_cols = build_matchup_frame(
        season_pairs,
        team_features_df[team_features_df["Season"] == season].copy(),
        feature_base_cols,
    )
    season_matchups["PredTeam1Won"] = model.predict(season_matchups[season_feature_cols])

    return pd.DataFrame({
        "WTeamID": np.where(
            season_matchups["PredTeam1Won"] == 1,
            season_matchups["Team1ID"],
            season_matchups["Team2ID"],
        ),
        "LTeamID": np.where(
            season_matchups["PredTeam1Won"] == 1,
            season_matchups["Team2ID"],
            season_matchups["Team1ID"],
        ),
    })


def score_bracket_for_season(season, predictions_df):
    bracket = Bracket(season, "M")
    bracket.seed()

    playin_cells = bracket.bracket.loc[bracket.bracket["playin"] == "a"]
    for i in range(len(playin_cells)):
        region = playin_cells.iloc[i]["region"]
        seed = playin_cells.iloc[i]["slot"]
        playins = bracket.get_cell(region, 0, seed)["pred"]
        bracket.set_predicted(
            region,
            1,
            bracket.get_slot(seed),
            bracket.get_winner(playins.iloc[0], playins.iloc[1], predictions_df),
        )
        bracket.set_correct(region, 1, bracket.get_slot(seed), "waiting")

    for region in bracket.regions:
        for rnd in range(1, 5):
            for slot in range(1, 2 ** (5 - rnd), 2):
                winner = bracket.get_winner(
                    bracket.get_predicted(region, rnd, slot),
                    bracket.get_predicted(region, rnd, slot + 1),
                    predictions_df,
                )
                bracket.set_predicted(region, rnd + 1, (slot + 1) // 2, winner)

    bracket.set_predicted(
        "W", 6, 1,
        bracket.get_winner(bracket.get_predicted("W", 5, 1), bracket.get_predicted("X", 5, 1), predictions_df),
    )
    bracket.set_predicted(
        "Y", 6, 1,
        bracket.get_winner(bracket.get_predicted("Y", 5, 1), bracket.get_predicted("Z", 5, 1), predictions_df),
    )
    bracket.set_predicted(
        "W", 7, 1,
        bracket.get_winner(bracket.get_predicted("W", 6, 1), bracket.get_predicted("Y", 6, 1), predictions_df),
    )

    bracket.add_results()
    with contextlib.redirect_stdout(io.StringIO()):
        score, _ = bracket.score()

    return int(score), bracket.get_team(bracket.get_predicted("W", 7, 1))


backtest_rows = []
for valid_season in [2023, 2024, 2025]:
    train_slice = tournament_matchups[tournament_matchups["Season"] < valid_season].copy()
    valid_slice = tournament_matchups[tournament_matchups["Season"] == valid_season].copy()

    backtest_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("tree", DecisionTreeClassifier(**decision_tree_params)),
    ])
    backtest_model.fit(train_slice[feature_cols], train_slice["Team1Won"])
    valid_slice["PredTeam1Won"] = backtest_model.predict(valid_slice[feature_cols])
    biased_valid_slice = apply_top_seed_bias(valid_slice, pred_col="PredTeam1Won")

    model_accuracy = accuracy_score(valid_slice["Team1Won"], valid_slice["PredTeam1Won"])
    biased_accuracy = accuracy_score(biased_valid_slice["Team1Won"], biased_valid_slice["BiasedPredTeam1Won"])

    bracket_matchups = build_matchup_frame(
        prediction_pairs[
            prediction_pairs["WTeamID"].isin(set(historical_team_features.loc[historical_team_features["Season"] == valid_season, "TeamID"]))
            & prediction_pairs["LTeamID"].isin(set(historical_team_features.loc[historical_team_features["Season"] == valid_season, "TeamID"]))
        ].assign(Season=valid_season),
        historical_team_features[historical_team_features["Season"] == valid_season].copy(),
        feature_base_cols,
    )[0]
    bracket_matchups["PredTeam1Won"] = backtest_model.predict(bracket_matchups[feature_cols])
    biased_bracket_matchups = apply_top_seed_bias(bracket_matchups, pred_col="PredTeam1Won")

    model_bracket_predictions = pd.DataFrame({
        "WTeamID": np.where(bracket_matchups["PredTeam1Won"] == 1, bracket_matchups["Team1ID"], bracket_matchups["Team2ID"]),
        "LTeamID": np.where(bracket_matchups["PredTeam1Won"] == 1, bracket_matchups["Team2ID"], bracket_matchups["Team1ID"]),
    })
    biased_bracket_predictions = pd.DataFrame({
        "WTeamID": np.where(biased_bracket_matchups["BiasedPredTeam1Won"] == 1, biased_bracket_matchups["Team1ID"], biased_bracket_matchups["Team2ID"]),
        "LTeamID": np.where(biased_bracket_matchups["BiasedPredTeam1Won"] == 1, biased_bracket_matchups["Team2ID"], biased_bracket_matchups["Team1ID"]),
    })

    model_bracket_score, model_champion = score_bracket_for_season(valid_season, model_bracket_predictions)
    biased_bracket_score, biased_champion = score_bracket_for_season(valid_season, biased_bracket_predictions)

    backtest_rows.append({
        "Season": valid_season,
        "ModelGameAccuracy": round(float(model_accuracy), 3),
        "SeedBiasedGameAccuracy": round(float(biased_accuracy), 3),
        "ModelBracketScore": model_bracket_score,
        "SeedBiasedBracketScore": biased_bracket_score,
        "ModelChampion": model_champion,
        "SeedBiasedChampion": biased_champion,
    })

backtest_results = pd.DataFrame(backtest_rows)
print("Recent walk-forward backtest")
display(backtest_results)

print("Average raw-tree bracket score across 2023-2025:", round(backtest_results["ModelBracketScore"].mean(), 2))
print("Average seed-biased bracket score across 2023-2025:", round(backtest_results["SeedBiasedBracketScore"].mean(), 2))

Recent walk-forward backtest


,Season,GameAccuracy,BracketScore,PredictedChampion
0,2023,0.687,52,Houston
1,2024,0.731,153,Connecticut
2,2025,0.791,148,Houston


Average bracket score across 2023-2025: 117.67
